In [22]:
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
from tqdm import tqdm

save_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/data/multigraph_manual")
metadata_dir = save_dir / "metadata"

subbasins = pd.read_parquet(metadata_dir / 'subbasins.parquet', columns=['site_id', 'nextdown','outlet_id'])
subbasins.index = subbasins.index.astype(str)
subbasins.reset_index(inplace=True)

subbasins

,site_id,nextdown,outlet_id
0,23021321,23017718,EAUF-A3750050
1,23021261,23020936,EAUF-A3750050
2,23021244,23021024,EAUF-A3750050
3,23021122,23020916,EAUF-A3750050
4,23021112,23020913,EAUF-A3750050
...,...,...,...
72188,81003765,81001413,USGS-15908000
72189,81001414,81001326,USGS-15908000
72190,81001413,81001326,USGS-15908000
72191,USGS-15908000,None,USGS-15908000


In [23]:
subbasins[subbasins['outlet_id'] == 'ABOM-10046010']

,site_id,nextdown,outlet_id
2633,ABOM-10046010,None,ABOM-10046010


In [25]:
G = nx.DiGraph()

# Add all subbasin IDs as nodes
G.add_nodes_from(subbasins['site_id'])

# Add edges for those with valid nextdown entries
edges = list(subbasins.loc[subbasins['nextdown'].notna(), ['site_id','nextdown']].itertuples(index=False, name=None))
G.add_edges_from(edges)

In [26]:
print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

# Find sources and sinks
sources = [n for n in G.nodes() if G.in_degree(n) == 0]
sinks = [n for n in G.nodes() if G.out_degree(n) == 0]
print(f"Sources (headwaters): {len(sources)}")
print(f"Sinks (outlets): {len(sinks)}")

Nodes: 72193
Edges: 70721
Sources (headwaters): 19210
Sinks (outlets): 1472


In [28]:
import json

graph_json = nx.readwrite.json_graph.node_link_data(G, edges='edges')
with open(metadata_dir / f'all_graphs.json', 'w') as f:
    json.dump(graph_json, f, indent=4)